# LAB GenAI - LLMs - OpenAI GPT API Exercises

## 0. Setup

In [1]:
import os
import json
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import tiktoken

pd.set_option("display.max_colwidth", 120)

load_dotenv()                                   # reads OPENAI_API_KEY from the .env file
print("Key found:", "OPENAI_API_KEY" in os.environ)   # never print the key itself

client = OpenAI()                               # finds the key in the environment
MODEL = "gpt-5.4-nano"                         

Key found: True


## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [10]:
def chatbot(messages, **options):
    """Send messages to the model and return the text of the reply."""
    if isinstance(messages, str):                      
        messages = [{"role": "user", "content": messages}]
    response = client.chat.completions.create(model=MODEL, 
                                              messages=messages, 
                                              **options)
    return [choice.message.content for choice in response.choices]

In [14]:
chatbot("Name the countries the red panda is native to."
        "List only the country names, separated by commas.")[0] # not adding other paarmeters

'Bhutan, China, India, Myanmar, Nepal'

In [15]:
# changing the number of response choices to three
answers= chatbot("Name the countries the red panda is native to."
                    "List only the country names, separated by commas.",
                      n=3)

for i, answer in enumerate(answers, start=1):
    print(f"Choice {i}: {answer}")

Choice 1: Bhutan, China, India, Myanmar, Nepal
Choice 2: Bhutan, China, India, Myanmar, Nepal
Choice 3: China, India, Nepal, Bhutan


Changing n changes the number of response choices.

In [18]:
# changing the temperature
for temperature in [0, 1.5]:
    answers= chatbot("Name the countries the red panda is native to."
                    "List only the country names, separated by commas.",
                      n=3, temperature= temperature)[0]
    print(f"temperature={temperature}:", answers)

temperature=0: China, India, Bhutan, Nepal, Myanmar
temperature=1.5: China, Nepal, Bhutan, India, Myanmar


The temperature change does not make a great change for this question. Let's try an other question.

In [19]:
for temperature in [0, 1.5]:
    answers= chatbot("Give me three names for a red panda."
                    "List only the names, separated by commas.",
                      n=3, temperature= temperature)[0]
    print(f"temperature={temperature}:", answers)

temperature=0: Mochi, Saffron, Ember
temperature=1.5: Pip, Saffron, Juniper


Again the temperature does not change that much, maybe I need to finde better questions.

In [23]:
# changing top p
for top_p in [0.1, 0.5, 1.0]:
    answers= chatbot("Give me names for a red panda."
                    "List only the names, separated by commas.",
                      n=3, top_p = top_p)[0]
    print(f"top_p ={top_p}:", answers)

top_p =0.1: Mochi, Saffron, Ember, Nori, Cinnamon, Hazel, Taro, Luna, Maple, Pepper, Kiki, Suki, Rusty, Juniper, Tinsel, Marigold, Basil, Poppy, Willow, Copper
top_p =0.5: Mochi, Saffron, Ember, Willow, Juniper, Copper, Luna, Koda, Maple, Hazel, Nori, Taro, Cinnamon, Pepper, Ruby, Suki, Tinsel, Marigold, Basil, Clover, Sable, Noodle, Penny, Ginger, Nova, Tofu, Rascal, Daisy, Cosmo, Poppy
top_p =1.0: Mochi, Saffron, Ember, Noodle, Pixie, Cinnamon, Marbles, Rusty, Lumen, Thimble, Jasper, Tika, Whiskers, Pepper, Sunny, Blaze, Maple, Juniper, Rolo, Waffles, Biscuit, Clover, Pudding, Copper, Zuzu


Top p mostly changed the number of names chosen. For the lowest p value the samllest number of names was chosen.

In [24]:
# changing the number of max tokens
for max_tokens in [5, 10, 15]:
    answers= chatbot("Name the countries the red panda is native to."
                    "List only the country names, separated by commas.",
                      n=3, max_completion_tokens = max_tokens)[0]
    print(f"max tokens={max_tokens}:", answers)

max tokens=5: Nepal
max tokens=10: China, Nepal, Bhutan, India
max tokens=15: Nepal, Bhutan, India, China, Myanmar


THe number of max tokens allows us to controll the length of the output.

In [26]:
# changing the frequenc penalty
for freq_pen in [0, 0.5, 1.5]:
    answers= chatbot("Write a small paragraph of about 20 words about red pandas. "
                        "Explain their habitat, diet, and threats. "
                        "Use natural language.",
                      n=3, frequency_penalty = freq_pen)[0]
    print(f"frequency penalty ={freq_pen}:", answers)

frequency penalty =0: Red pandas live in cool, misty mountain forests of the Himalayas, eating bamboo, fruits, and small insects, but face habitat loss and poaching.
frequency penalty =0.5: Red pandas live in cool mountain forests with dense bamboo, feeding on bamboo leaves, berries, and insects. Habitat loss threatens them.
frequency penalty =1.5: Red pandas live in cool, forested mountains of the Himalayas, eating bamboo and fruit. Logging threatens them with habitat loss.


Increasing the frequency penalty changes the structure of the phrases and of the text, as certain tokens will be repeated less often.

In [27]:
# changing the presence penalty
for pres_pen in [0, 0.5, 1.5]:
    answers= chatbot("Write a small paragraph of about 20 words about red pandas. "
                        "Explain their habitat, diet, and threats. "
                        "Use natural language.",
                      n=3, presence_penalty = freq_pen)[0]
    print(f"presence penalty = {pres_pen}:", answers)

presence penalty = 0: Red pandas live in cool, misty forests of the Himalayas, eating bamboo, leaves, fruits, and insects. Habitat loss threatens them.
presence penalty = 0.5: Red pandas live in cool temperate forests of the Himalayas, eating bamboo, berries, and insects, but face habitat loss and poaching.
presence penalty = 1.5: Red pandas live in cool Himalayan forests, feeding on bamboo, fruits, and insects. They’re threatened by habitat loss and poaching.


Increasing the presence penalty changes the structure of the text even more, as ceratin tokones won't be repeated.

In [28]:
# changing stop
for stop_sequence in [None, "END"]:
    answers = chatbot("Give three facts about red pandas. "
                    "After each fact, write END.",
                    n=3,
                    stop=stop_sequence)[0]

    print(f"\nstop={stop_sequence}:\n{answers}")


stop=None:
1) Red pandas have a specialized “false thumb” that helps them grip bamboo and other foods. END  
2) They are mainly found in the Himalayas and nearby forests in Nepal, India, Bhutan, and China. END  
3) Red pandas eat mostly bamboo, but they also sometimes eat berries, eggs, mushrooms, and insects. END


BadRequestError: Error code: 400 - Unsupported parameter: 'stop' is not supported with this model.

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [32]:
def summarize(text, **options):
    """Summarise the text input into a few sentences"""
    messages = [{"role": "user",
                 "content": f"Summarize the following text in a maximum of 150 "
                            f"characters, including spaces. Do not exceed this limit.\n\n"
                            f"{text}"}]
    response = client.chat.completions.create(model=MODEL, 
                                              messages=messages, 
                                              **options)
    return [choice.message.content for choice in response.choices]

In [33]:
text = """
The red panda (Ailurus fulgens), also known as the lesser panda, is a small mammal native to the 
eastern Himalayas and southwestern China. It has dense reddish-brown fur with a black belly and legs, 
white-lined ears, a mostly white muzzle and a ringed tail. Its head-to-body length is 51–63.5 cm 
(20.1–25.0 in) with a 28–48.5 cm (11.0–19.1 in) tail, and it weighs between 3.2 and 15 kg (7.1 and 33.1 lb). 
It is well adapted to climbing due to its flexible joints and curved semi-retractile claws.
"""

summary = summarize(text)
print(summary[0])

Red panda: small Himalayan/China mammal with reddish-brown fur, ringed tail, and climbing adaptations via flexible joints and semi-retractile claws.


In [34]:
# changing the temperature
for temperature in [0, 1.5]:
    summary= summarize(text,
                      n=3, temperature= temperature)[0]
    print(f"temperature={temperature}:", summary)

temperature=0: Red panda: small Himalayan mammal with reddish fur, ringed tail, white muzzle/ears; 51–63.5 cm, 3.2–15 kg; agile climber.
temperature=1.5: Red panda: small Himalayan mammal with reddish-brown fur, ringed tail, flexible joints for climbing; 51–63.5 cm, 3.2–15 kg.


In [37]:
# changing the number of max tokens
for max_tokens in [5, 10, 15]:
    summary= summarize(text,
                              n=3, 
                              max_completion_tokens = max_tokens)[0]
    print(f"max tokens={max_tokens}:", summary)

max tokens=5: Red panda
max tokens=10: Red panda: small Himalayan mammal
max tokens=15: Red panda: small Himalayan/China mammal with reddish fur


The limitation of the number of tokens is very vissble for this example.

In [38]:
# changing top p
for top_p in [0.1, 0.5, 1.0]:
    summary = summarize (text,
                      n=1, top_p = top_p)[0]
    print(f"top_p ={top_p}:", summary)

top_p =0.1: Red panda: small Himalayan mammal with reddish fur, ringed tail, white muzzle/ears; 51–63.5 cm, 3.2–15 kg; agile climber.
top_p =0.5: Red panda: small Himalayan mammal with reddish-brown fur, ringed tail, and semi-retractile claws for climbing.
top_p =1.0: Red panda: small Himalayan/China mammal with reddish fur, ringed tail, and semi-retractile claws adapted for climbing. 51–63.5 cm.


In [39]:
# changing the frequency penalty
for freq_pen in [0, 0.5, 1.5]:
    summary = summarize(text,
                      n=1, frequency_penalty = freq_pen)[0]
    print(f"frequency penalty ={freq_pen}:", summary)

frequency penalty =0: Red panda: small Himalayan mammal with reddish-brown fur, ringed tail; 51–63.5 cm long, 3.2–15 kg; agile climber.
frequency penalty =0.5: Red panda: Himalayan–China mammal with reddish fur, ringed tail; 51–63.5 cm long, 3.2–15 kg; agile climber with semi-retractile claws.
frequency penalty =1.5: Red panda: small Himalayan mammal with reddish fur, ringed tail, and curved claws for climbing. 51–63.5 cm long; 3.2–15 kg.


In [40]:
# changing the presence penalty
for pres_pen in [0, 0.5, 1.5]:
    summary = summarize(text,
                      n=1, presence_penalty = freq_pen)[0]
    print(f"presence penalty = {pres_pen}:", summary)

presence penalty = 0: Red panda: small Himalayan mammal with reddish fur, ringed tail; adapted for climbing via flexible joints and semi-retractile claws.
presence penalty = 0.5: Red panda: small Himalayan mammal with reddish-brown fur, ringed tail; adapted for climbing via flexible joints and semi-retractile claws.
presence penalty = 1.5: Red panda: small Himalayan mammal with reddish fur, ringed tail; 51–63.5 cm long, 3.2–15 kg; adapted for climbing.


The parameter best_of can not be used with OpenAI's Chat Completions API, wich was used to create the function.
- best_of generates n candidates completions on the server and returns the one with the highest average log probability per token 

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user",
            "content": ("Summarize this text in no more than 150 characters:\n\n"f"{text}")}],
                n=1,
                logprobs=True,
                top_logprobs=3)

print(response.choices[0].message.content)

Red panda: small eastern Himalaya mammal with reddish fur, ringed tail, and semi-retractile claws for climbing.


## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [64]:
def translate_text(text, source_language, target_language, **options):
    """Translate text from one language to another."""

    messages = [{"role": "user",
                "content": (f"Translate the following text from {source_language} "
                        f"into {target_language}. "
                        "Return only the translation.\n\n"
                        f"Text: {text}")}]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **options)

    return [choice.message.content for choice in response.choices]

In [65]:
text = "The red panda inhabits coniferous forests as well as temperate broadleaf and mixed forests, favouring steep slopes with dense bamboo cover close to water sources."
source = "English"
target = "German"

result = translate_text( text, source, target,n=1)

print(result[0])

Der Rotpanda bewohnt sowohl Nadelwälder als auch gemäßigte Laubwälder und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.


In [66]:
# changing the temperature
for temperature in [0, 1.5]:
    result = translate_text( text, source, target,n=1, temperature= temperature)[0]
    print(f"temperature={temperature}:", result)

temperature=0: Der Rotpanda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
temperature=1.5: Der Rote Panda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs nahe an Wasserquellen.


In [67]:
# changing the number of max tokens
for max_tokens in [5, 10, 15]:
    result = translate_text( text, source, target,n=1, max_completion_tokens = max_tokens)[0]
    print(f"max tokens={max_tokens}:", result)

max tokens=5: Der Rot
max tokens=10: Der Rotpanda lebt in N
max tokens=15: Der Rotpanda bewohnt Nadelwälder


For a translation limiting the number of tokens does not make much sense.

In [68]:
# changing top p
for top_p in [0.1, 0.5, 1.0]:
    result = translate_text( text, source, target,
                      n=1, top_p = top_p)[0]
    print(f"top_p ={top_p}:", result)

top_p =0.1: Der Rotpanda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
top_p =0.5: Der Rotpanda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
top_p =1.0: Der Rotpanda bewohnt Nadelwälder sowie temperierte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.


In [69]:
# changing the frequenc penalty
for freq_pen in [0, 0.5, 1.5]:
    result = translate_text( text, source, target,n=1, frequency_penalty = freq_pen)[0]
    print(f"frequency penalty ={freq_pen}:", result)

frequency penalty =0: Der Rote Panda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
frequency penalty =0.5: Der Roten Panda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
frequency penalty =1.5: Der Rotpanda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.


In [70]:
# changing the presence penalty
for pres_pen in [0, 0.5, 1.5]:
    result = translate_text( text, source, target,n=1, presence_penalty = freq_pen)[0]
    print(f"presence penalty = {pres_pen}:", result)

presence penalty = 0: Der Rotpanda lebt in Nadelwäldern sowie in gemäßigten Laub- und Mischwäldern und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
presence penalty = 0.5: Der Rotpanda bewohnt sowohl Nadelwälder als auch gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.
presence penalty = 1.5: Der Rotpanda bewohnt Nadelwälder sowie gemäßigte Laub- und Mischwälder und bevorzugt steile Hänge mit dichtem Bambusbewuchs in der Nähe von Wasserquellen.


`echo` is not supported by the Chat Completions API. It belongs to the legacy Completions API.

In [71]:
# changing the likelihood of specific tokens
for bias in [0, 5, 10]:
    result = translate_text(
        text, source, target,
        n=1,logit_bias={"TOKEN_ID": bias})[0]

    print(f"Logit bias = {bias}:", result)

BadRequestError: Error code: 400 - Unsupported parameter: 'logit_bias' is not supported with this model.

`logit_bias` is not supported by the Chat Completions API. It belongs to the legacy Completions API.

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [42]:
def sentiment_analyses(text, **options):
    """Determine the sentiment of a given text."""

    messages = [{"role": "user",
                "content": ("Classify the sentiment of the following text as "
                            "one of these labels: positive, negative, neutral. "
                            "Return only the correct label.\n\n"
                            f"Text: {text}")}]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **options)

    return [choice.message.content for choice in response.choices]

In [43]:
text = "Red pandas are very cute! Their fur has an extraodinary red color."
result = sentiment_analyses(text, temperature=0, n=1)

print(result[0])

positive


In [44]:
# changing the temperature
for temperature in [0, 1.5]:
    sentiment = sentiment_analyses(text,
                      n=1, temperature= temperature)[0]
    print(f"temperature={temperature}:", sentiment)

temperature=0: positive
temperature=1.5: positive


In [45]:
# changing the number of max tokens
for max_tokens in [5, 10, 15]:
    sentiment = sentiment_analyses(text,
                      n=1, max_completion_tokens = max_tokens)[0]
    print(f"max tokens={max_tokens}:", sentiment)

max tokens=5: positive
max tokens=10: positive
max tokens=15: positive


In [47]:
# changing top p
for top_p in [0.1, 0.5, 1.0]:
    sentiment = sentiment_analyses(text,
                      n=1, top_p = top_p)[0]
    print(f"top_p ={top_p}:", sentiment)

top_p =0.1: positive
top_p =0.5: positive
top_p =1.0: positive


In [48]:
# changing the frequenc penalty
for freq_pen in [0, 0.5, 1.5]:
    sentiment = sentiment_analyses(text,
                      n=1, frequency_penalty = freq_pen)[0]
    print(f"frequency penalty ={freq_pen}:", sentiment)

frequency penalty =0: positive
frequency penalty =0.5: positive
frequency penalty =1.5: positive


In [49]:
# changing the presence penalty
for pres_pen in [0, 0.5, 1.5]:
    sentiment = sentiment_analyses(text,
                      n=1, presence_penalty = freq_pen)[0]
    print(f"presence penalty = {pres_pen}:", sentiment)

presence penalty = 0: positive
presence penalty = 0.5: positive
presence penalty = 1.5: positive


In [51]:
# changing the number of response choices to three
sentiment = sentiment_analyses(text,
                      n=3)

for i, sentiment in enumerate(sentiment, start=1):
    print(f"Choice {i}: {sentiment}")

Choice 1: positive
Choice 2: positive
Choice 3: positive


In [52]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[ { "role": "user",
            "content": (
                "Classify the sentiment of the following text as "
                "positive, negative, or neutral. "
                "Return only the label.\n\n"
                f"{text}")}],
                n=1,
                logprobs=True,
                top_logprobs=3)

print(response.choices[0].message.content)

positive


## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [53]:
def text_completion(text, **options):
    """Generate a short text based on a given prompt."""
    messages = [
        {"role": "user",
            "content": ("Continue the following prompt by generating a short text. "
                        "The response must not exceed 100 characters, including spaces.\n\n"
                        f"Prompt: {text}")}]

    response = client.chat.completions.create(model=MODEL, messages=messages, **options)

    return [choice.message.content for choice in response.choices]

In [56]:
text_comp = ("Once upon a time, a small red panda")
result = text_completion( text_comp, n=1)

print(result[0])

found a map in bamboo and followed it to a hidden stream.


In [57]:
# changing the temperature
for temperature in [0, 1.5]:
    result = text_completion (text_comp,
                      n=1, temperature= temperature)[0]
    print(f"temperature={temperature}:", text_comp, result)

temperature=0: Once upon a time, a small red panda found a lost acorn and shared it with friends.
temperature=1.5: Once upon a time, a small red panda nibble-d apple. At dawn, it found a new friend.


In [59]:
# changing the number of max tokens
for max_tokens in [5, 10, 15]:
    result = text_completion (text_comp,
                          n=1, max_completion_tokens = max_tokens)[0]
    print(f"max tokens={max_tokens}:", text_comp, result)

max tokens=5: Once upon a time, a small red panda found a
max tokens=10: Once upon a time, a small red panda found a glowing acorn and wished
max tokens=15: Once upon a time, a small red panda bravely climbed a moonlit tree and shared berries with


In [60]:
# changing top p
for top_p in [0.1, 0.5, 1.0]:
    result = text_completion (text_comp,
                              n=1, top_p = top_p)[0]
    print(f"top_p ={top_p}:", text_comp, result)

top_p =0.1: Once upon a time, a small red panda found a lost acorn and shared it with friends.
top_p =0.5: Once upon a time, a small red panda found a lost maple leaf and wished for adventure.
top_p =1.0: Once upon a time, a small red panda found a mysterious leaf that glowed softly at night


In [61]:
# changing the frequenc penalty
for freq_pen in [0, 0.5, 1.5]:
    result = text_completion (text_comp,
                              n=1, frequency_penalty = freq_pen)[0]
    print(f"frequency penalty ={freq_pen}:", text_comp, result)

frequency penalty =0: Once upon a time, a small red panda who loved rainy nights and danced on bamboo leaves.
frequency penalty =0.5: Once upon a time, a small red panda curled up under bamboo, dreaming of distant, moonlit mountains.
frequency penalty =1.5: Once upon a time, a small red panda found a lost kite, tied it to her tail, and smiled all night.


In [62]:
# changing the presence penalty
for pres_pen in [0, 0.5, 1.5]:
    result = text_completion (text_comp,
                              n=1, presence_penalty = freq_pen)[0]
    print(f"presence penalty = {pres_pen}:", text_comp,result)

presence penalty = 0: Once upon a time, a small red panda found a lost lantern, glowing softly in the forest at dusk.
presence penalty = 0.5: Once upon a time, a small red panda who loved mangoes, but always hid them for later.
presence penalty = 1.5: Once upon a time, a small red panda strolled through bamboo, dreaming of moonlit adventures.


# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)